In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")


In [ ]:
import sys
import subprocess

# Install latest numpy version using pip
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "numpy"])


In [2]:
import numpy as np
import pandas as pd
import os

## load population matrix


In [3]:
num_files = 8  # Change this to the actual number of files to read

age_matrix_vec = []

path = "/Users/haolong/Documents/demos_v_matrix/output/"
for i in range(num_files):
    # Create the filename dynamically

    filename = f"popu_matrix_{i}.csv"
    matrix = pd.read_csv(path + filename).to_numpy() # convert to numpy
    age_matrix_vec.append(matrix)

for idx in range(8):
    matrix = age_matrix_vec[idx]  # Extract the matrix
    # Check each row, and keep it if not all values are negative
    filtered_matrix = matrix[~np.all(matrix < 0, axis=1)]
    age_matrix_vec[idx] = filtered_matrix

## load albu matrix

In [ ]:
# Load and verify matrices
albu_1_mat_storage = []
FLAG = 2023
save_dir = '../data/new_2023/albu_matrix'
for idx in range(8):
    file_path = os.path.join(save_dir, f"albu_mat_{FLAG}_{idx}.npy")
    matrix = np.load(file_path)
    albu_1_mat_storage.append(matrix)
    print(f"Loaded: {file_path}, Shape: {matrix.shape}")



## load bmi, hyer and diabetes


In [7]:
N = 8
parent_dir = "../data/bmi_matrix" 
bmi_matrix_ls = []
for idx in range(N):
    file_path = os.path.join(parent_dir, f"bmi_matrix_{idx}.npy")
    matrix = np.load(file_path)
    bmi_matrix_ls.append(matrix)

In [8]:
parent_dir = "../data/bmi_matrix" 
hyper_mat_ls_v = []
for idx in range(N):
    file_path = os.path.join(parent_dir, f"hyper_mat_{idx}.npy")
    matrix = np.load(file_path)
    hyper_mat_ls_v.append(matrix)

In [9]:
parent_dir = "../data/bmi_matrix" 
diabetes_mat_ls_v = []
for idx in range(N):
    file_path = os.path.join(parent_dir, f"diabetes_mat_{idx}.npy")
    matrix = np.load(file_path)
    diabetes_mat_ls_v.append(matrix)

In [ ]:
diabetes_mat_ls_v[0]

In [ ]:
# Initialize cumulative counters
total_pre_diabetes_count = 0
total_population_count = 0

for idx, diabetes_matrix in enumerate(diabetes_mat_ls_v):
    # Extract the second-to-last column (pre-diabetes status)
    second_last_column = diabetes_matrix[:, -2]
    last_col_ages = age_matrix_vec[idx][:, -2]

    # Create mask for age range 18-74
    mask_all = (last_col_ages >= 18) & (last_col_ages <= 74)

    # Apply mask to filter relevant data
    filtered_data = second_last_column[mask_all]

    # Update cumulative counts
    total_pre_diabetes_count += np.sum(filtered_data == 0.5)
    total_population_count += np.sum(mask_all)

# Compute final pre-diabetes prevalence
final_pre_diabetes_percentage = (total_pre_diabetes_count / total_population_count) * 100 if total_population_count > 0 else 0

# Output the final prevalence
print(f"Final Pre-Diabetes Prevalence: {final_pre_diabetes_percentage:.2f}%")


In [ ]:
import numpy as np

# Initialize cumulative counters
diabetes_count = 0
total_population_count = 0

for idx, diabetes_matrix in enumerate(diabetes_mat_ls_v):
    # Extract the second-to-last column (pre-diabetes status)
    diabetes_status = diabetes_matrix[:, -2]
    ages = age_matrix_vec[idx][:, -2]

    # Create mask for age range 18-74
    mask = (ages >= 18) & (ages <= 74)

    # Apply mask to filter relevant data
    filtered_data = diabetes_status[mask]

    # Update cumulative counts
    diabetes_count += np.sum(filtered_data == 1)
    total_population_count += np.sum(mask)

# Compute final pre-diabetes prevalence
final_diabetes_percentage = (diabetes_count / total_population_count) * 100 if total_population_count > 0 else 0

# Output the final prevalence
print(f"Final diabetes Prevalence: {final_diabetes_percentage:.2f}%")



In [ ]:
ages = age_matrix_vec[0][:, -2]
mask = (ages >= 18) & (ages <= 74)
hyper_mat_ls_v[0][mask, -2].shape

In [ ]:
np.sum(mask)

In [ ]:
25728 / 57984

In [ ]:
diabetes_mat_ls_v[0].shape[0]

## check hypertension and diabetes , draw a Venn diagram




In [ ]:
!pip install --upgrade matplotlib-venn

In [18]:
from matplotlib_venn import venn2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

def calculate_percentages_and_plot(diabetes_mat_ls_v, hyper_mat_ls_v):
    total_people = 0
    diabetes_counts = 0
    hyper_counts = 0
    both_counts = 0
    
    for idx in range(len(diabetes_mat_ls_v)):  # Iterate over all available indices
        ages = age_matrix_vec[idx][:, -2]
        mask = (ages >= 18) & (ages <= 74)
        total_people += np.sum(mask)  # Count total people
        diabetes_counts += np.sum(diabetes_mat_ls_v[idx][mask, -2] == 1)  # Count diabetes cases
        hyper_counts += np.sum(hyper_mat_ls_v[idx][mask, -2])  # Count hypertension cases
        print( f'idx {idx},hyper_counts {hyper_counts} ')
        both_counts += np.sum( (diabetes_mat_ls_v[idx][mask, -2] == 1) * hyper_mat_ls_v[idx][mask, -2])  # Count both conditions

    # Compute percentages
    diabetes_percentage = round((diabetes_counts / total_people) * 100, 2)
    hyper_percentage = round((hyper_counts / total_people) * 100, 2)
    both_percentage = round((both_counts / total_people) * 100, 2)

    # Print results
    print(f"Diabetes Prevalence: {diabetes_percentage:.2f}%")
    print(f"Hypertension Prevalence: {hyper_percentage:.2f}%")
    print(f"Both Conditions Prevalence: {both_percentage:.2f}%")

    plt.figure(figsize=(5, 5))
    venn = venn2(subsets=(diabetes_percentage, hyper_percentage, both_percentage), 
                    set_labels=('Diabetes', 'Hypertension'))
    plt.title("Overlap of Diabetes and Hypertension (%)")

    # Adjust label positions
    for text in venn.set_labels:
        if text:
            text.set_fontsize(12)
            text.set_horizontalalignment('center')

    for text in venn.subset_labels:
        if text:
            text.set_fontsize(10)
            text.set_horizontalalignment('center')

    plt.show()



# Example call (ensure diabetes_mat_ls_v and hyper_mat_ls_v are defined before calling)
calculate_percentages_and_plot(diabetes_mat_ls_v, hyper_mat_ls_v)


## map ckd 

In [ ]:
import pandas as pd

# Read only the first 10 rows (lines 0 to 9) from the Excel file
# ckd/data/mcmc_parameters.xlsx ../data/
 
file_path = '../data/mcmc_parameters.xlsx'
data = pd.read_excel(file_path, nrows=10)

# Extract the relevant coefficients for each ethnicity
ethnicities = ['chn', 'mal', 'ind']
coefficients = {}

for eth in ethnicities:
    # Get the row for the specific ethnicity and convert it to a vector
    coefficients[eth] = data.loc[data['eth'] == eth, ['beta_0', 'beta_1', 'beta_2', 'beta_3', 'beta_4','sigma']].values.flatten()

# Display the coefficients
print(coefficients)
#

In [ ]:
# Compute eGFR values
        eGFR_matrix = (
            beta0 +
            beta1 * matrix * 0.5 +  # Age contribution
            beta2 * gender2 +  # Gender contribution
            beta3 * matrix * gender2 * 0.5 +  # Age * Gender interaction
            beta4 * np.log(bmi_values) +  # BMI contribution
            beta1 * matrix * hyper_coefficient * hyper_value + 
            beta1 * matrix * diabetes_coefficient * diabete_value + 
            beta1 * matrix * albu_coefficient * albu_value + 
            row_noise  # Random noise
        )

In [ ]:
import numpy as np

coefficients = {
    'chn mal': np.array([130.84 , -0.4671, 14.9112, 0, -5.3749 , 13.3742], dtype=object),
    'chn fem':np.array([130.84, -0.4671, 14.9112, -0.2066, -5.3749, 13.3742], dtype=object),
    'mal mal': np.array([121.98, -0.3741, 18.7098, 0, -4.5513, 14.2774], dtype=object),
    'mal fem': np.array([121.98, -0.3741, 18.7098, -0.2948, -4.5513, 14.2774], dtype=object),
    'ind mal': np.array([118.31, -0.3034, 19.8534, 0, -4.6831, 14.4905], dtype=object),
    'ind fem': np.array([118.31, -0.3034, 19.8534, -0.2, -4.6831 , 14.4905], dtype=object),
    
}

print(coefficients)


## use negative

In [2380]:

def map_eth_str(idx):
    ethnic_keys = ['chn mal','chn fem','mal mal','mal fem','ind mal','ind fem','mal mal','mal fem']  # Mapping indices to dictionary keys
    return ethnic_keys[idx ]

## with albuminuria 


In [ ]:
albu_1_mat_storage[idx][0,:,:]

In [ ]:
albu_1_mat_storage[0].shape

In [ ]:
import numpy as np

eGFR_matrix_ls = []  # List to store eGFR matrices

for idx in range(8):
  # Iterate over the two cases
    case_matrices = []  # Store matrices for this case
    for i, k in enumerate([0, 20]):
        # Extract age matrix for this index
        matrix = age_matrix_vec[idx]  # Contains age values
        bmi_values = bmi_matrix_ls[idx]  # Corresponding BMI values
        albu_value = albu_1_mat_storage[idx][k, :, :]
        diabete_value = diabetes_mat_ls_v[idx]
        hyper_value = hyper_mat_ls_v[idx]
        
        eGFR_matrix = np.zeros_like(matrix, dtype=float)
        
        # Get ethnicity and gender
        eth = map_eth_str(idx)  # 'chn', 'ind', or 'mal'
        gender2 = idx % 2  # 0 for Male, 1 for Female

        # Extract coefficients for the ethnicity
        beta0, beta1, beta2, beta3, beta4, sigma = coefficients[eth]
        
        # Generate noise
        row_noise = np.random.normal(loc=0, scale=sigma, size=matrix.shape)
        
        # Compute coefficients based on conditions
        diabetes_coefficient = np.where(diabete_value == 0.5, 0.5, np.where(diabete_value == 1, 1., 0))
        albu_coefficient = np.where(albu_value == 1, 0.1, np.where(albu_value == 2, 0.5, 0))
        hyper_coefficient = 0.1

        # Compute eGFR values
        eGFR_matrix = (
            beta0 +
            beta1 * matrix * 0.5 +  # Age contribution
            beta2 * gender2 +  # Gender contribution
            beta3 * matrix * gender2 * 0.5 +  # Age * Gender interaction
            beta4 * np.log(bmi_values) +  # BMI contribution
            beta1 * matrix * hyper_coefficient * hyper_value + 
            beta1 * matrix * diabetes_coefficient * diabete_value + 
            beta1 * matrix * albu_coefficient * albu_value + 
            row_noise  # Random noise
        )
        
        # Replace invalid age values with -1 in eGFR matrix
        eGFR_matrix[matrix == -1] = -1
        
        case_matrices.append(eGFR_matrix)
    case_matrices = np.array(case_matrices)
    
    eGFR_matrix_ls.append(case_matrices)

# Convert list to a NumPy array for better structure


# Display the resulting eGFR matrix
print("eGFR Matrix:")
print(eGFR_matrix_ls[0][0,:,:])


In [ ]:
print("eGFR_matrix shape:", eGFR_matrix.shape)
print("matrix shape:", matrix.shape)

In [ ]:
eGFR_matrix.shape

In [ ]:
eGFR_matrix_ls[0].shape

## stages mapping

In [2435]:
stages = {
    1: lambda x: x > 90,
    2: lambda x: (x >= 60) & (x < 90),
    3.1: lambda x: (x >= 45) & (x < 60),
    3.2: lambda x: (x >= 30) & (x < 45),
    4 : lambda x: (x >= 15) & (x < 30),
    5 : lambda x: (x >= 0) & (x < 15),
    -1 :lambda x: x == -1
}

In [2436]:
# For each matrix, create a corresponding stage matrix (same shape) where each element gets the stage value.
stage_matrix_ls = []
for i in range(8):
    matrix = eGFR_matrix_ls[i][-1]
    # Initialize an output matrix with NaNs
    stage_matrix = np.full(matrix.shape, np.nan)
    # For every stage, update the positions where the condition is True
    for stage, condition in stages.items():
        mask = condition(matrix)
        stage_matrix[mask] = stage
    stage_matrix_ls.append(stage_matrix)

In [ ]:
stage_matrix_ls[0].shape

In [2438]:
stage_keys = [1, 2, 3.1, 3.2, 4, 5, -1]

# Initialize an 8 x 7 matrix to hold the counts for each of the 8 matrices and 7 stages
stage_counts_mat = np.zeros((8, len(stage_keys)), dtype=int)

# Iterate over the 8 matrices
for i in range(8):
    # Extract the age values from the last column of age_matrix_vec[i]
    ages = age_matrix_vec[i][:, -2]
    # Similarly, extract the corresponding stage values from the last column of stage_matrix_ls[i]
    # (assuming stage values are aligned row-wise)
    stages_col = stage_matrix_ls[i][:, -2]
    
    # Create a mask for entries with age between 18 and 74 (inclusive)
    age_mask = (ages >= 18) & (ages <= 74)
    
    # Apply the mask to the stage values
    filtered_stages = stages_col[age_mask]
    
    # Count how many entries fall into each stage
    counts = [np.sum(filtered_stages == stage) for stage in stage_keys]
    stage_counts_mat[i, :] = counts

In [2439]:
col_counts = stage_counts_mat.sum(axis=0)

# Total entries across all stages
total_entries = col_counts.sum()

# Compute percentages for each stage (if total_entries is nonzero)
percentages = (col_counts / total_entries * 100)

In [ ]:
percentages

## ckd table mapping


In [ ]:
albu_1_mat_storage[i].shape

In [ ]:
import numpy as np
import pandas as pd

# Define GFR stages corresponding to the table rows
stage_keys = [1, 2, 3.1, 3.2, 4, 5]  # G1, G2, G3a, G3b, G4, G5

# Define ACR categories corresponding to the table columns
acr_categories = [0, 1, 2]  # Corresponding to A1, A2, A3

# Initialize an empty DataFrame for counts
df_counts = pd.DataFrame(0, index=stage_keys, columns=acr_categories)

# Iterate over the 8 groups
for i in range(8):
    # Extract relevant data
    ages = age_matrix_vec[i][:, -2]   # Age column
    stages_col = stage_matrix_ls[i][:, -2]  # GFR stages
    albu_values = albu_1_mat_storage[i][-1, :, -2]  # Albumin levels for different thresholds
    
    # Filter ages in range [18, 74]
    age_mask = (ages >= 18) & (ages <= 74)
    
    # Apply the mask to get valid stages
    filtered_stages = stages_col[age_mask]
    filtered_acr = albu_values[age_mask]
    # Iterate over albumin levels (ACR categories: 0,1,2 → A1,A2,A3)
    for acr_idx, acr_level in enumerate(acr_categories):
        # Count occurrences per stage and update df_counts
        for stage in stage_keys:
            count = np.sum((filtered_stages == stage) & (filtered_acr == acr_level))
            df_counts.loc[stage, acr_level] += count

# Rename columns for clarity
df_counts.columns = ['A1', 'A2', 'A3']
df_counts.index = ['G1', 'G2', 'G3a', 'G3b', 'G4', 'G5']

# Display the final table
print(df_counts)



In [ ]:
df_counts.sum()

In [ ]:
# Compute the total sum of all counts
total_sum = df_counts.sum().sum()
 

# Convert counts to percentages based on the total sum
df_percent = (df_counts / total_sum) * 100

# Display the percentage table
print(df_percent)



In [ ]:
# Compute row sums and add as a new column
df_percent["Total"] = df_percent.sum(axis=1)

# Compute column sums and add as a new row
df_percent.loc["Total"] = df_percent.sum(axis=0)

# Display the updated DataFrame
print(df_percent)


# upper bound

import numpy as np
import pandas as pd

# Define GFR stages corresponding to the table rows
stage_keys = [1, 2, 3.1, 3.2, 4, 5]  # G1, G2, G3a, G3b, G4, G5

# Define ACR categories corresponding to the table columns
acr_categories = [0, 1, 2]  # Corresponding to A1, A2, A3

# Initialize an empty DataFrame for counts
df_counts_upperbound = pd.DataFrame(0, index=stage_keys, columns=acr_categories)

# Iterate over the 8 groups
for i in range(8):
    # Extract relevant data
    ages = age_matrix_vec[i][:, -2]   # Age column
    stages_col = stage_matrix_ls[i][:, -2]  # GFR stages
    albu_values = albu_1_mat_storage[i][4, :, -2]  # Albumin levels for different thresholds
    
    # Filter ages in range [18, 74]
    age_mask = (ages >= 18) & (ages <= 74)
    
    # Apply the mask to get valid stages
    filtered_stages = stages_col[age_mask]
    filtered_acr = albu_values[age_mask]
    # Iterate over albumin levels (ACR categories: 0,1,2 → A1,A2,A3)
    for acr_idx, acr_level in enumerate(acr_categories):
        # Count occurrences per stage and update df_counts
        for stage in stage_keys:
            count = np.sum((filtered_stages == stage) & (filtered_acr == acr_level))
            df_counts.loc[stage, acr_level] += count

# Rename columns for clarity
df_counts.columns = ['A1', 'A2', 'A3']
df_counts.index = ['G1', 'G2', 'G3a', 'G3b', 'G4', 'G5']

# Display the final table
print(df_counts)



In [ ]:
# Compute the total sum of all counts
total_sum = df_counts.sum().sum()
 

# Convert counts to percentages based on the total sum
df_percent = (df_counts / total_sum) * 100



# Compute row sums and add as a new column
df_percent["Total"] = df_percent.sum(axis=1)

# Compute column sums and add as a new row
df_percent.loc["Total"] = df_percent.sum(axis=0)

# Display the updated DataFrame
print(df_percent)

In [ ]:


# Initialize DataFrames for counts
df_counts = pd.DataFrame(0, index=stage_keys, columns=acr_categories)
df_counts_upperbound = pd.DataFrame(0, index=stage_keys, columns=acr_categories)

# Iterate over 8 groups for df_counts
for i in range(8):
    ages = age_matrix_vec[i][:, -2]
    stages_col = stage_matrix_ls[i][:, -2]
    acr_values = albu_1_mat_storage[i][0, :, -2]  # ACR for base case
    
    age_mask = (ages >= 18) & (ages <= 74)
    
    filtered_stages = stages_col[age_mask]
    filtered_acr = acr_values[age_mask]
    
    for acr_level in acr_categories:
        for stage in stage_keys:
            count = np.sum((filtered_stages == stage) & (filtered_acr == acr_level))
            df_counts.loc[stage, acr_level] += count

# Iterate over 8 groups for df_counts_upperbound
for i in range(8):
    ages = age_matrix_vec[i][:, -2]
    stages_col = stage_matrix_ls[i][:, -2]
    acr_values_upper = albu_1_mat_storage[i][-1, :, -2]  # ACR for upper bound
    
    age_mask = (ages >= 18) & (ages <= 74)
    
    filtered_stages = stages_col[age_mask]
    filtered_acr = acr_values_upper[age_mask]
    
    for acr_level in acr_categories:
        for stage in stage_keys:
            count = np.sum((filtered_stages == stage) & (filtered_acr == acr_level))
            df_counts_upperbound.loc[stage, acr_level] += count

# Convert counts to percentages using TOTAL sum
df_perc = df_counts / df_counts.values.sum() * 100
df_perc_upperbound = df_counts_upperbound / df_counts_upperbound.values.sum() * 100

# Rename columns for readability
df_counts.columns = df_counts_upperbound.columns = ['A1', 'A2', 'A3']
df_perc.columns = df_perc_upperbound.columns = ['A1', 'A2', 'A3']
df_counts.index = df_counts_upperbound.index = ['G1', 'G2', 'G3a', 'G3b', 'G4', 'G5']
df_perc.index = df_perc_upperbound.index = ['G1', 'G2', 'G3a', 'G3b', 'G4', 'G5']

# Format final table (x1, x2) where x1 from df_counts & df_perc, x2 from df_counts_upperbound & df_perc_upperbound
df_final = df_counts.astype(str) + " (" + df_perc.round(2).astype(str) + "%), " + df_counts_upperbound.astype(str) + " (" + df_perc_upperbound.round(2).astype(str) + "%)"

# Add row and column sums
df_final["Toal"] = df_counts.sum(axis=1).astype(str) + " (" + df_perc.sum(axis=1).round(2).astype(str) + "%), " + df_counts_upperbound.sum(axis=1).astype(str) + " (" + df_perc_upperbound.sum(axis=1).round(2).astype(str) + "%)"
df_final.loc["Total"] = df_counts.sum(axis=0).astype(str) + " (" + df_perc.sum(axis=0).round(2).astype(str) + "%), " + df_counts_upperbound.sum(axis=0).astype(str) + " (" + df_perc_upperbound.sum(axis=0).round(2).astype(str) + "%)"

# Display the final table
print(df_final)


In [ ]:
# Function to format percentages correctly
def format_percentages(perc_df, perc_upper_df):
    formatted_df = perc_df.copy()
    for row in formatted_df.index:
        for col in formatted_df.columns:
            v1 = round(perc_df.loc[row, col], 2)
            v2 = round(perc_upper_df.loc[row, col], 2)
            formatted_df.loc[row, col] = f"({min(v1, v2)}, {max(v1, v2)})"
    return formatted_df

# Apply formatting function
df_final = format_percentages(df_perc, df_perc_upperbound)

# Add row and column sums
df_final["Row Sum"] = format_percentages(df_perc.sum(axis=1).to_frame(), df_perc_upperbound.sum(axis=1).to_frame()).iloc[:, 0]
df_final.loc["Column Sum"] = format_percentages(df_perc.sum(axis=0).to_frame().T, df_perc_upperbound.sum(axis=0).to_frame().T).iloc[0]

# Display the final table
print(df_final)

In [2449]:
df_final.to_csv('results.csv')

# calculate the percentage within diabetes and hypertension

In [ ]:
import pandas as pd
import numpy as np

# Initialize counts for diabetes, pre-diabetes, and non-diabetes groups
ckd_counts_with_diabetes = np.zeros((2, 2))  # (lower bound, upper bound)
ckd_counts_pre_diabetes = np.zeros((2, 2))
ckd_counts_without_diabetes = np.zeros((2, 2))

# Iterate for k = 0 (lower bound) and k = -1 (upper bound)
for idx, k in enumerate([0, -1]):  
    for i in range(8):
        # Extract relevant matrices
        ages = age_matrix_vec[i][:, -2]
        diabetes_status = diabetes_mat_ls_v[i][:, -2]
        acr_values = albu_1_mat_storage[i][k, :, -2]  # ACR for k=0 (lower) and k=-1 (upper)
        stages_col = stage_matrix_ls[i][:, -2]
        
        # Create age mask for ages between 18 and 74
        age_mask = (ages >= 18) & (ages <= 74)

        # Filter the stages, ACR values, and diabetes status
        filtered_stages = stages_col[age_mask]
        filtered_acr = acr_values[age_mask]
        filtered_diabetes_status = diabetes_status[age_mask]
        
        # Healthy condition: stage 0,1 and ACR value = 0
        healthy_mask = (filtered_stages <= 2) & (filtered_acr == 0)
        
        # Total without healthy cases (i.e., CKD)
        non_healthy_mask = ~healthy_mask
        
        # Count CKD prevalence in diabetes group (status = 1)
        ckd_counts_with_diabetes[idx, 0] += np.sum((filtered_diabetes_status == 1) & non_healthy_mask)
        ckd_counts_with_diabetes[idx, 1] += np.sum(filtered_diabetes_status == 1)
        
        # Count CKD prevalence in pre-diabetes group (status = 0.5)
        ckd_counts_pre_diabetes[idx, 0] += np.sum((filtered_diabetes_status == 0.5) & non_healthy_mask)
        ckd_counts_pre_diabetes[idx, 1] += np.sum(filtered_diabetes_status == 0.5)

        # Count CKD prevalence in non-diabetes group (status = 0)
        ckd_counts_without_diabetes[idx, 0] += np.sum((filtered_diabetes_status == 0) & non_healthy_mask)
        ckd_counts_without_diabetes[idx, 1] += np.sum(filtered_diabetes_status == 0)

# Convert counts to percentages safely
def safe_percentage(numerator, denominator):
    return (numerator / denominator * 100) if denominator > 0 else np.nan

# Compute lower (k=0) and upper (k=-1) bound percentages
ckd_perc_with_diabetes = [
    safe_percentage(ckd_counts_with_diabetes[0, 0], ckd_counts_with_diabetes[0, 1]),
    safe_percentage(ckd_counts_with_diabetes[1, 0], ckd_counts_with_diabetes[1, 1])
]
ckd_perc_pre_diabetes = [
    safe_percentage(ckd_counts_pre_diabetes[0, 0], ckd_counts_pre_diabetes[0, 1]),
    safe_percentage(ckd_counts_pre_diabetes[1, 0], ckd_counts_pre_diabetes[1, 1])
]
ckd_perc_without_diabetes = [
    safe_percentage(ckd_counts_without_diabetes[0, 0], ckd_counts_without_diabetes[0, 1]),
    safe_percentage(ckd_counts_without_diabetes[1, 0], ckd_counts_without_diabetes[1, 1])
]

# Create final matrix with percentage range
final_matrix = pd.DataFrame({
    'Diabetes': [f"({min(ckd_perc_with_diabetes):.2f}%, {max(ckd_perc_with_diabetes):.2f}%)"],
    'Pre-Diabetes': [f"({min(ckd_perc_pre_diabetes):.2f}%, {max(ckd_perc_pre_diabetes):.2f}%)"],
    'No Diabetes': [f"({min(ckd_perc_without_diabetes):.2f}%, {max(ckd_perc_without_diabetes):.2f}%)"]
}, index=['Estimated Percentage'])

# Add NPHS 2022 Prevalence row
final_matrix.loc["NPHS 2022 Prevalence"] = ["42.3%", "21.8%", "10.0%"]

# Display the final table
print(final_matrix)


In [2451]:
# print(final_matrix)

#                               Diabetes      Pre-Diabetes      No Diabetes
# Estimated Percentage  (39.63%, 46.13%)  (10.61%, 18.43%)  (6.99%, 12.80%)
# NPHS 2022 Prevalence             42.3%             21.8%            10.0%

# -----
# I wish to plot estiamteed Percentage as a bar, and then plot NPHS 2022 Prevalence  
# we want a plot that shows the estimated percentage and the NPHS 2022 Prevalence in diabetes and no diabetes, no need for pre diabets version

# give me code 

In [ ]:
import matplotlib.pyplot as plt

# Categories (using categorical positions for tighter layout)
labels = ['Diabetes', 'No Diabetes']
x = range(len(labels))

# Estimated ranges
estimated_lows = [39.63, 6.99]
estimated_highs = [46.13, 12.80]

# NPHS values
nphs_values = [42.3, 10.0]

# Plotting
fig, ax = plt.subplots(figsize=(4, 6))  # Narrower width

# Plot estimated range as vertical lines
for i in x:
    ax.plot([i, i], [estimated_lows[i], estimated_highs[i]],
            color='skyblue', linewidth=4, label='Estimated 95% CI' if i == 0 else "")

# Plot NPHS prevalence as red dots
ax.scatter(x, nphs_values, color='red', zorder=2, label='NPHS 2022 Prevalence', s=30)

# Customizing x-axis
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlim(-0.5, len(labels) - 0.5)  # Tighter x-limits
ax.set_ylabel('Percentage (%)')
ax.set_title('Estimated vs NPHS 2022 CKD Prevalence by diabetes status')
ax.legend()
ax.set_ylim(0, 55)
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
ckd_counts_with_diabetes

In [2454]:
# 15.7% pre diabetes

In [ ]:
import pandas as pd
import numpy as np

# Initialize counts for hypertension and non-hypertension groups
ckd_counts_with_hypertension = np.zeros(2)
ckd_counts_without_hypertension = np.zeros(2)

# Initialize percentages for hypertension and non-hypertension groups
ckd_perc_with_hypertension = np.zeros(2)
ckd_perc_without_hypertension = np.zeros(2)

# Iterate over the 8 groups to count CKD prevalence within hypertension and without hypertension
for i in range(8):
    # Extract relevant matrices
    ages = age_matrix_vec[i][:, -2]
    hypertension_status = hyper_mat_ls_v[i][:, -2]  # Hypertension status matrix
    acr_values = albu_1_mat_storage[i][-1, :, -2]  # ACR for base case
    stages_col = stage_matrix_ls[i][:, -2]
    
    # Create age mask for ages between 18 and 74
    age_mask = (ages >= 18) & (ages <= 74)

    # Filter the stages and ACR values
    filtered_stages = stages_col[age_mask]
    filtered_acr = acr_values[age_mask]
    filtered_hypertension_status = hypertension_status[age_mask]
    
    # Healthy condition: stage 0,1 and ACR value = 0
    healthy_mask = (filtered_stages <= 2) & (filtered_acr == 0)
    
    # Total without healthy cases (i.e., CKD)
    non_healthy_mask = ~healthy_mask
    
    # Count CKD prevalence in hypertension group
    ckd_in_hypertension = np.sum((filtered_hypertension_status == 1) & non_healthy_mask)
    ckd_counts_with_hypertension[0] += ckd_in_hypertension
    ckd_counts_with_hypertension[1] += np.sum(filtered_hypertension_status == 1)
    
    # Count CKD prevalence in non-hypertension group
    ckd_without_hypertension = np.sum((filtered_hypertension_status == 0) & non_healthy_mask)
    ckd_counts_without_hypertension[0] += ckd_without_hypertension
    ckd_counts_without_hypertension[1] += np.sum(filtered_hypertension_status == 0)

# Convert counts to percentages
ckd_perc_with_hypertension[0] = (ckd_counts_with_hypertension[0] / ckd_counts_with_hypertension[1]) * 100
ckd_perc_without_hypertension[0] = (ckd_counts_without_hypertension[0] / ckd_counts_without_hypertension[1]) * 100

# Create final 2x1 matrix for hypertension and non-hypertension groups
final_matrix_hypertension = pd.DataFrame({
    'Hypertension': [f"{ckd_counts_with_hypertension[0]} ({ckd_perc_with_hypertension[0]:.2f}%)"],
    'No Hypertension': [f"{ckd_counts_without_hypertension[0]} ({ckd_perc_without_hypertension[0]:.2f}%)"]
}, index=['CKD Percentage'])

# Display the final 2x1 matrix
print(final_matrix_hypertension)


In [ ]:
import pandas as pd
import numpy as np

# Initialize counts for hypertension and non-hypertension groups (lower and upper bounds)
ckd_counts_with_hypertension = np.zeros((2, 2))  # (lower bound, upper bound)
ckd_counts_without_hypertension = np.zeros((2, 2))

# Iterate for k = 0 (lower bound) and k = -1 (upper bound)
for idx, k in enumerate([0, -1]):  
    for i in range(8):
        # Extract relevant matrices
        ages = age_matrix_vec[i][:, -2]
        hypertension_status = hyper_mat_ls_v[i][:, -2]  # Hypertension status matrix
        acr_values = albu_1_mat_storage[i][k, :, -2]  # ACR for k=0 (lower) and k=-1 (upper)
        stages_col = stage_matrix_ls[i][:, -2]
        
        # Create age mask for ages between 18 and 74
        age_mask = (ages >= 18) & (ages <= 74)

        # Filter the stages and ACR values
        filtered_stages = stages_col[age_mask]
        filtered_acr = acr_values[age_mask]
        filtered_hypertension_status = hypertension_status[age_mask]
        
        # Healthy condition: stage 0,1 and ACR value = 0
        healthy_mask = (filtered_stages <= 2) & (filtered_acr == 0)
        
        # Total without healthy cases (i.e., CKD)
        non_healthy_mask = ~healthy_mask
        
        # Count CKD prevalence in hypertension group
        ckd_counts_with_hypertension[idx, 0] += np.sum((filtered_hypertension_status == 1) & non_healthy_mask)
        ckd_counts_with_hypertension[idx, 1] += np.sum(filtered_hypertension_status == 1)
        
        # Count CKD prevalence in non-hypertension group
        ckd_counts_without_hypertension[idx, 0] += np.sum((filtered_hypertension_status == 0) & non_healthy_mask)
        ckd_counts_without_hypertension[idx, 1] += np.sum(filtered_hypertension_status == 0)

# Convert counts to percentages safely
def safe_percentage(numerator, denominator):
    return (numerator / denominator * 100) if denominator > 0 else np.nan

# Compute lower (k=0) and upper (k=-1) bound percentages
ckd_perc_with_hypertension = [
    safe_percentage(ckd_counts_with_hypertension[0, 0], ckd_counts_with_hypertension[0, 1]),
    safe_percentage(ckd_counts_with_hypertension[1, 0], ckd_counts_with_hypertension[1, 1])
]
ckd_perc_without_hypertension = [
    safe_percentage(ckd_counts_without_hypertension[0, 0], ckd_counts_without_hypertension[0, 1]),
    safe_percentage(ckd_counts_without_hypertension[1, 0], ckd_counts_without_hypertension[1, 1])
]

# Create final matrix with percentage range
final_matrix_hypertension = pd.DataFrame({
    'Hypertension': [f"({min(ckd_perc_with_hypertension):.2f}%, {max(ckd_perc_with_hypertension):.2f}%)"],
    'No Hypertension': [f"({min(ckd_perc_without_hypertension):.2f}%, {max(ckd_perc_without_hypertension):.2f}%)"]
}, index=['Estimated Percentage'])

# Add NPHS 2022 Prevalence row
final_matrix_hypertension.loc["NPHS 2022 Prevalence"] = ["24.2%", "6.9%"]

# Display the final table
print(final_matrix_hypertension)


In [ ]:
import matplotlib.pyplot as plt

# Categories
labels = ['Hypertension', 'No Hypertension']
x = range(len(labels))

# Estimated ranges
estimated_lows = [14.6, 4]
estimated_highs = [28.7, 14.84]

# NPHS values
nphs_values = [24.2, 6.9]

# Plotting
fig, ax = plt.subplots(figsize=(4, 6))  # Compact width

# Plot estimated range as vertical lines
for i in x:
    ax.plot([i, i], [estimated_lows[i], estimated_highs[i]],
            color='skyblue', linewidth=4, label='Estimated 95% CI' if i == 0 else "")

# Plot NPHS prevalence as red dots
ax.scatter(x, nphs_values, color='red', zorder=5, label='NPHS 2022 Prevalence', s=30)

# Customizing x-axis
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlim(-0.5, len(labels) - 0.5)
ax.set_ylim(0, 55)
ax.set_ylabel('Percentage (%)')
ax.set_title('Estimated vs NPHS 2022 CKD Prevalence by hypertension status')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


## overall prevalence


In [2458]:
import numpy as np

In [ ]:
age_matrix_vec[i].shape

In [ ]:
import numpy as np

# Initialize total counts
total_population = 0
total_ckd_cases = 0


# Iterate over 8 ethnicity groups
for i in range(8):
    # Extract age, diabetes, ACR, and stage matrices
    ages = age_matrix_vec[i][:, -2]
    acr_values = albu_1_mat_storage[i][0, :, -2]  # ACR for base case
    stages_col = stage_matrix_ls[i][:, -2]

    # Apply age mask
    age_mask = (ages >= 18) & (ages <= 74)

    # Filter variables based on the mask
    filtered_stages = stages_col[age_mask]
    filtered_acr = acr_values[age_mask]

    # Define healthy condition: stage 0 or 1 and ACR = 0
    healthy_mask = (filtered_stages <=2) & (filtered_acr == 0)

    # Identify non-healthy (CKD) cases
    non_healthy_mask = ~healthy_mask

    # Count total people and CKD cases in this ethnicity group
    total_population += np.sum(age_mask)
    total_ckd_cases += np.sum(non_healthy_mask)

# Compute overall prevalence
overall_ckd_prevalence = (total_ckd_cases / total_population) * 100

# Print result
print(f"Overall CKD Prevalence (Ages 18-74): {total_ckd_cases} / {total_population} ({overall_ckd_prevalence:.2f}%)")


In [ ]:
albu_1_mat_storage[i][1, :, :]

In [ ]:
albu_1_mat_storage[i][0, :, :]

In [ ]:

# Initialize total counts


# Initialize lists to store prevalence for each scenario
prevalence_k0 = []
prevalence_k1 = []

# Iterate through each year (columns -34 to -2)
for year_col in range(-34, -1):
    # For k=0 and k=1 scenarios
    for k in [0, 20]:
        total_population = 0
        total_ckd_cases = 0

        # Iterate over 8 ethnicity groups
        for i in range(8):
            # Extract age, diabetes, ACR, and stage matrices
            ages = age_matrix_vec[i][:, year_col]
            acr_values = albu_1_mat_storage[i][k, :, year_col]  # ACR for scenario k
            stages_col = stage_matrix_ls[i][:, year_col]

            # Apply age mask
            age_mask = (ages >= 18) & (ages <= 74)

            # Filter variables based on the mask
            filtered_stages = stages_col[age_mask]
            filtered_acr = acr_values[age_mask]

            # Define healthy condition: stage 0 or 1 and ACR = 0
            healthy_mask = (filtered_stages <=2) & (filtered_acr == 0)

            # Identify non-healthy (CKD) cases
            non_healthy_mask = ~healthy_mask

            # Count total people and CKD cases in this ethnicity group
            total_population += np.sum(age_mask)
            total_ckd_cases += np.sum(non_healthy_mask)

        # Compute overall prevalence
        overall_ckd_prevalence = (total_ckd_cases / total_population) * 100
        
        # Store in appropriate list based on k value
        if k == 0:
            prevalence_k0.append(overall_ckd_prevalence)
        else:
            prevalence_k1.append(overall_ckd_prevalence)

# Create dataframe with both scenarios
prevalence_df = pd.DataFrame({
    'k0_prevalence': prevalence_k0,
    'k1_prevalence': prevalence_k1
})

# Print result
print(f"Overall CKD Prevalence (Ages 18-74): {total_ckd_cases} / {total_population} ({overall_ckd_prevalence:.2f}%)")

In [ ]:
# Set initial and final years
start_year = 2000  # Can be changed to 2000 or other year
end_year = 2022

# Create years array
years = np.arange(start_year, end_year + 1)
num_years = len(years)

# Create figure and axis
plt.figure(figsize=(10, 6))

# Plot the confidence interval as a shaded region
plt.fill_between(years, prevalence_df['k0_prevalence'][-num_years:], 
                 prevalence_df['k1_prevalence'][-num_years:], 
                 alpha=0.2, color='lightblue', label='95% CI')

# Plot the mean line
mean_prevalence = (prevalence_df['k0_prevalence'][-num_years:] + 
                  prevalence_df['k1_prevalence'][-num_years:])/2
plt.plot(years, mean_prevalence, color='#6495ED', linewidth=2, label='Mean Prevalence')

# Add dots for mean prevalence values
plt.plot(years, mean_prevalence, 'o', color='darkblue', markersize=6)

# Add specific data points
# plt.plot([2019], [8.7], 'o', color='red', markersize=8, label='2020 Data Point')
# plt.plot([2022], [13.8], 'o', color='red', markersize=8, label='2022 Data Point')

# Customize the plot
plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('CKD Prevalence (%)', fontsize=12, fontweight='bold')
plt.title(f'CKD Prevalence (Ages 18-74) with 95% CI ({start_year}-{end_year})',
         fontsize=14, fontweight='bold', pad=15)

# Add grid
plt.grid(True, linestyle='--', alpha=0.3, axis='y')

# Customize ticks
plt.xticks(years[::2])  # Show every 2nd year
plt.tick_params(axis='both', labelsize=10)
plt.ylim(0,25)
# Add legend
plt.legend(loc='upper left', fontsize=12)

# Adjust layout
plt.tight_layout()

# Show plot
plt.show()



In [ ]:
# Create years array from 2010 to 2022 
years = np.arange(2010, 2023)

# Create figure and axis
plt.figure(figsize=(10, 6))

# Plot the confidence interval as a shaded region
plt.fill_between(years, prevalence_df['k0_prevalence'][-13:], prevalence_df['k1_prevalence'][-13:], 
                 alpha=0.2, color='lightblue', label='95% CI')

# Plot the mean line
mean_prevalence = (prevalence_df['k0_prevalence'][-13:] + prevalence_df['k1_prevalence'][-13:])/2
plt.plot(years, mean_prevalence, color='#6495ED', linewidth=2, label='Mean Prevalence')

# Add dots for mean prevalence values
plt.plot(years, mean_prevalence, 'o', color='darkblue', markersize=6)

# Add specific data points
# plt.plot([2019], [8.7], 'o', color='red', markersize=8, label='2020 Data Point')
# plt.plot([2022], [13.8], 'o', color='red', markersize=8, label='2022 Data Point')


# Customize the plot
plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('CKD Prevalence (%)', fontsize=12, fontweight='bold')
plt.title('CKD Prevalence (Ages 18-74) with 95% CI (2010-2022)',
         fontsize=14, fontweight='bold', pad=15)

# Add grid
plt.grid(True, linestyle='--', alpha=0.3, axis='y')

# Customize ticks
plt.xticks(years[::2])  # Show every 2nd year
plt.tick_params(axis='both', labelsize=10)
plt.ylim(0,25)
# Add legend
plt.legend(loc='upper left', fontsize=12)

# Adjust layout
plt.tight_layout()

# Show plot
plt.show()



In [ ]:
overall_ckd_prevalence

In [2467]:
# 13.2 - 14.59

## demo

In [ ]:
import numpy as np

# Initialize lists to store results
ethnicity_ckd_counts = []
ethnicity_populations = []
ethnicity_ckd_prevalences = []

# Iterate over 8 ethnicity groups
for i in range(8):
    # Extract age, diabetes, ACR, and stage matrices
    ages = age_matrix_vec[i][:, -2]
    acr_values = albu_1_mat_storage[i][0, :, -2]  # ACR for base case
    stages_col = stage_matrix_ls[i][:, -2]

    # Apply age mask
    age_mask = (ages >= 18) & (ages <= 74)

    # Filter variables based on the mask
    filtered_stages = stages_col[age_mask]
    filtered_acr = acr_values[age_mask]

    # Define healthy condition: stage 0 or 1 and ACR = 0
    healthy_mask = (filtered_stages <= 2) & (filtered_acr == 0)

    # Identify non-healthy (CKD) cases
    non_healthy_mask = ~healthy_mask

    # Count total people and CKD cases in this ethnicity group
    total_population = np.sum(age_mask)
    total_ckd_cases = np.sum(non_healthy_mask)

    # Compute prevalence
    ckd_prevalence = (total_ckd_cases / total_population) * 100 if total_population > 0 else 0

    # Store results
    ethnicity_populations.append(total_population)
    ethnicity_ckd_counts.append(total_ckd_cases)
    ethnicity_ckd_prevalences.append(ckd_prevalence)

# Print results
for i in range(8):
    print(f"Ethnicity {i}: {ethnicity_ckd_counts[i]} / {ethnicity_populations[i]} ({ethnicity_ckd_prevalences[i]:.2f}%)")


In [2469]:
import seaborn as sns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Define ethnicity categories
categories = ['Chinese\nMale', 'Chinese\nFemale', 'Malay\nMale', 'Malay\nFemale', 'Indian\nMale', 'Indian\nFemale']
nphs_prevalence = [11.8, 13.8, 20.7, 17.2, 17.6, 11.0]  # Target values

# Define colors for each ethnicity
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# Initialize lists for storing results
results = []

# Iterate over 6 ethnicity groups
for i in range(6):
    ages = age_matrix_vec[i][:, -2]
    age_mask = (ages >= 18) & (ages <= 74)
    total_population = np.sum(age_mask)

    # Compute CKD prevalence for k=2 and k=18
    bounds = []
    for k in [1, 19]:
        acr_values = albu_1_mat_storage[i][k, :, -2]  # ACR for current case
        stages_col = stage_matrix_ls[i][:, -2]

        # Filter variables based on the mask
        filtered_stages = stages_col[age_mask]
        filtered_acr = acr_values[age_mask]

        # Define healthy condition: stage 0 or 1 and ACR = 0
        healthy_mask = (filtered_stages <= 2) & (filtered_acr == 0)

        # Identify CKD cases
        total_ckd_cases = np.sum(~healthy_mask)

        # Compute prevalence
        prevalence = (total_ckd_cases / total_population) * 100 if total_population > 0 else 0
        bounds.append(prevalence)

    # Store results for this ethnicity
    results.append({
        "Ethnicity": categories[i],
        "Lower": min(bounds),
        "Upper": max(bounds),
        "Color": colors[i]
    })

# Set Seaborn style for publication-quality plot
sns.set_theme(style="white", font_scale=1.5)

# Create the plot
plt.figure(figsize=(10, 6))

# Plot vertical lines for each ethnicity with different colors
for result in results:
    plt.vlines(x=result["Ethnicity"], ymin=result["Lower"], ymax=result["Upper"], 
               color=result["Color"], linewidth=2)

# Overlay target prevalence values with star markers
plt.scatter(categories, nphs_prevalence, color="grey", marker='x', s=200, label="2022 Prevalence (NPHS)", zorder=3)

# Add custom legend entry for vertical lines showing 95% CI
plt.vlines(x=[], ymin=[], ymax=[], color='black', linewidth=2, label='Estimated 95% CI')

# Customize plot appearance
plt.xlabel("Demographic Groups", y=-2.05)
plt.ylabel("CKD Prevalence (%)")
plt.title("CKD Prevalence Range Across Demographic Groups (Ages 18-74)", fontsize=16, fontweight='bold')
plt.legend()



plt.ylim(0, 75)  # Set y-axis limits
plt.grid(True, alpha=0.3, axis='y')

# Improve layout and show plot
plt.tight_layout()
plt.show()


import numpy as np
import pandas as pd

# Initialize lists to store results
results = []

# Iterate over 6 ethnicity groups
for i in range(6):
    # Extract age and stage matrices
    ages = age_matrix_vec[i][:, -2]

    # Apply age mask
    age_mask = (ages >= 18) & (ages <= 74)

    total_population = np.sum(age_mask)
    
    # Compute CKD cases and prevalence for each k-value
    for k in range(21):  # k from 0 to 4
        acr_values = albu_1_mat_storage[i][k, :, -2]  # ACR for current case
        stages_col = stage_matrix_ls[i][:, -2]

        # Filter variables based on the mask
        filtered_stages = stages_col[age_mask]
        filtered_acr = acr_values[age_mask]

        # Define healthy condition: stage 0 or 1 and ACR = 0
        healthy_mask = (filtered_stages <= 2) & (filtered_acr == 0)

        # Identify CKD cases
        total_ckd_cases = np.sum(~healthy_mask)

        # Compute prevalence
        prevalence = (total_ckd_cases / total_population) * 20 if total_population > 0 else 0

        # Store results
        results.append({
            "Ethnicity": i + 1,  # Assuming ethnicity is indexed from 1 to 6
            "k_value": k,
            "Total_Population": total_population,
            "CKD_Cases": total_ckd_cases,
            "Prevalence (%)": prevalence
        })

# Create DataFrame
df_results = pd.DataFrame(results)

# Display the DataFrame
print(df_results)


In [ ]:
import numpy as np

# Initialize lists to store results
ethnicity_ckd_counts = []
ethnicity_populations = []
ethnicity_ckd_prevalences = []
k = -1 
# k = 0 and k = -1 
# Iterate over 8 ethnicity groups
for i in range(8):
    # Extract age, diabetes, ACR, and stage matrices
    ages = age_matrix_vec[i][:, -2]
    acr_values = albu_1_mat_storage[i][k, :, -2]  # ACR for base case
    stages_col = stage_matrix_ls[i][:, -2]

    # Apply age mask
    age_mask = (ages >= 18) & (ages <= 74)

    # Filter variables based on the mask
    filtered_stages = stages_col[age_mask]
    filtered_acr = acr_values[age_mask]

    # Define healthy condition: stage 0 or 1 and ACR = 0
    healthy_mask = (filtered_stages <= 2) & (filtered_acr ==0)

    # Identify non-healthy (CKD) cases
    non_healthy_mask = ~healthy_mask

    # Count total people and CKD cases in this ethnicity group
    total_population = np.sum(age_mask)
    total_ckd_cases = np.sum(non_healthy_mask)

    # Compute prevalence
    ckd_prevalence = (total_ckd_cases / total_population) * 100 if total_population > 0 else 0

    # Store results
    ethnicity_populations.append(total_population)
    ethnicity_ckd_counts.append(total_ckd_cases)
    ethnicity_ckd_prevalences.append(ckd_prevalence)

# Print results
for i in range(8):
    print(f"Ethnicity {i}: {ethnicity_ckd_counts[i]} / {ethnicity_populations[i]} ({ethnicity_ckd_prevalences[i]:.2f}%)")

categories = ['chn mal', 'chn fem', 'mal mal', 'mal fem', 'ind mal', 'ind fem']
nphs_prevalence  = [11.8, 13.8, 20.7, 17.2, 17.6, 11.0]


import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Initialize lists to store results
ethnicity_ckd_counts_low = []
ethnicity_ckd_counts_high = []
ethnicity_populations = []
ethnicity_ckd_prevalences_low = []
ethnicity_ckd_prevalences_high = []

# Iterate over 8 ethnicity groups
for i in range(6):
    # Extract age and stage matrices
    ages = age_matrix_vec[i][:, -2]

    # Apply age mask
    age_mask = (ages >= 18) & (ages <= 74)

    total_population = np.sum(age_mask)
    
    # Compute lower and upper bound
    ckd_cases_low, ckd_cases_high = [], []

    for k in range(21):  # k=0 for lower bound, k=-1 for upper bound
        acr_values = albu_1_mat_storage[i][k, :, -2]  # ACR for current case
        stages_col = stage_matrix_ls[i][:, -2]

        # Filter variables based on the mask
        filtered_stages = stages_col[age_mask]
        filtered_acr = acr_values[age_mask]

        # Define healthy condition: stage 0 or 1 and ACR = 0
        healthy_mask = (filtered_stages <= 2) & (filtered_acr == 0)

        # Identify CKD cases
        total_ckd_cases = np.sum(~healthy_mask)

        # Store CKD counts
        if k == -1:
            ckd_cases_low.append(total_ckd_cases)
        else:
            ckd_cases_high.append(total_ckd_cases)

    # Compute prevalence
    prevalence_low = (ckd_cases_low[0] / total_population) * 100 if total_population > 0 else 0
    prevalence_high = (ckd_cases_high[0] / total_population) * 100 if total_population > 0 else 0

    # Store results
    ethnicity_populations.append(total_population)
    ethnicity_ckd_counts_low.append(ckd_cases_low[0])
    ethnicity_ckd_counts_high.append(ckd_cases_high[0])
    ethnicity_ckd_prevalences_low.append(prevalence_low)
    ethnicity_ckd_prevalences_high.append(prevalence_high)

# Print results
for i in range(6):
    print(f"Ethnicity {i}: {ethnicity_ckd_counts_low[i]}-{ethnicity_ckd_counts_high[i]} / {ethnicity_populations[i]} "
          f"({ethnicity_ckd_prevalences_low[i]:.2f}-{ethnicity_ckd_prevalences_high[i]:.2f}%)")

# Data for plotting
group_labels = ["Chn Male", "Chn Female", "Malay Male", "Malay Female", "Ind Male", "Ind Female"]
data = {
    "Age Range": group_labels,
    "Low Prevalence": ethnicity_ckd_prevalences_low,
    "Upper Prevalence": ethnicity_ckd_prevalences_high,
    "Target": [11.8, 13.8, 20.7, 17.2, 17.6, 11.0]  # Change this if you want a different target
}



boxplot_data = pd.DataFrame({
    "Demographic Group": ['chn mal', 'chn fem', 'mal mal', 'mal fem', 'ind mal', 'ind fem'],
    "Lower Prevalence": ethnicity_ckd_prevalences_low,
    "Upper Prevalence": ethnicity_ckd_prevalences_high
})

df_melted = boxplot_data.melt(id_vars=["Demographic Group"], 
                              value_vars=["Lower Prevalence", "Upper Prevalence"], 
                              var_name="Bound", value_name="Prevalence")

nphs_prevalence  = [11.8, 13.8, 20.7, 17.2, 17.6, 11.0]
categories = ['chn mal', 'chn fem', 'mal mal', 'mal fem', 'ind mal', 'ind fem']
# Plot
plt.figure(figsize=(8, 5))
sns.set_style("whitegrid")

# Boxplot for CKD prevalence range
sns.boxplot(x="Demographic Group", y="Prevalence", data=df_melted, width=0.5, palette="Blues")

# Overlay the target prevalence as points
sns.stripplot(x=categories, y=nphs_prevalence, color="red", 
              marker="o", size=8, label="Target Prevalence", jitter=False)

# Labels and title
plt.ylim(0, 70)
plt.xlabel("Demographic Group")
plt.ylabel("Prevalence (%)")
plt.title("Prevalence of CKD by Demographic Group")
plt.legend()

# Show plot
plt.show()

In [2472]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define ethnicity categories
categories = ['chn mal', 'chn fem', 'mal mal', 'mal fem', 'ind mal', 'ind fem']
nphs_prevalence = [11.8, 13.8, 20.7, 17.2, 17.6, 11.0]  # Target values

# Initialize lists for storing results
results = []

# Iterate over 6 ethnicity groups
for i in range(6):
    ages = age_matrix_vec[i][:, -2]
    age_mask = (ages >= 18) & (ages <= 74)
    total_population = np.sum(age_mask)

    # Compute CKD prevalence for k-values 0 to 4
    for k in range(5):
        acr_values = albu_1_mat_storage[i][k, :, -2]  # ACR for current case
        stages_col = stage_matrix_ls[i][:, -2]

        # Filter variables based on the mask
        filtered_stages = stages_col[age_mask]
        filtered_acr = acr_values[age_mask]

        # Define healthy condition: stage 0 or 1 and ACR = 0
        healthy_mask = (filtered_stages <= 2) & (filtered_acr == 0)

        # Identify CKD cases
        total_ckd_cases = np.sum(~healthy_mask)

        # Compute prevalence
        prevalence = (total_ckd_cases / total_population) * 100 if total_population > 0 else 0

        # Duplicate values by 10 times
        for _ in range(10):
            results.append({"Ethnicity": categories[i], "Prevalence (%)": prevalence})




In [ ]:
# Set Seaborn style for publication-quality plot with pure white background
sns.set_theme(style="white", font_scale=1.5)

# Create the figure and axis
plt.figure(figsize=(10, 6))

# Create the box plot
ax = sns.boxplot(x="Ethnicity", y="Prevalence (%)", data=df_results, palette="pastel", zorder=1)

# Overlay target prevalence values with 'x' markers, ensuring they are on top
plt.scatter(categories, nphs_prevalence, color="red", marker='x', s=100, label="Target Prevalence (NPHS)", zorder=2)

# Customize plot appearance
plt.xlabel("")
plt.ylabel("CKD Prevalence (%)")
plt.xticks(rotation=0)  # Rotate x-axis labels for readability
plt.ylim(0, 30)  # Set y-axis limits
plt.title("Comparison of CKD Prevalence Across Ethnicities", fontsize=16, fontweight='bold')
plt.legend()

# Improve layout and show plot
plt.tight_layout()
plt.show()


In [ ]:
boxplot_data = pd.DataFrame({
    "Demographic Group": ['chn mal', 'chn fem', 'mal mal', 'mal fem', 'ind mal', 'ind fem'],
    "Lower Prevalence": ethnicity_ckd_prevalences_low,
    "Upper Prevalence": ethnicity_ckd_prevalences_high
})

df_melted = boxplot_data.melt(id_vars=["Demographic Group"], 
                              value_vars=["Lower Prevalence", "Upper Prevalence"], 
                              var_name="Bound", value_name="Prevalence")

nphs_prevalence  = [11.8, 13.8, 20.7, 17.2, 17.6, 11.0]
categories = ['chn mal', 'chn fem', 'mal mal', 'mal fem', 'ind mal', 'ind fem']
# Plot
plt.figure(figsize=(8, 5))
sns.set_style("whitegrid")

# Boxplot for CKD prevalence range
sns.boxplot(x="Demographic Group", y="Prevalence", data=df_melted, width=0.5, palette="Blues")

# Overlay the target prevalence as points
sns.stripplot(x=categories, y=nphs_prevalence, color="red", 
              marker="o", size=8, label="Target Prevalence", jitter=False)

# Labels and title
plt.ylim(0, 70)
plt.xlabel("Demographic Group")
plt.ylabel("Prevalence (%)")
plt.title("Prevalence of CKD by Demographic Group")
plt.legend()

# Show plot
plt.show()

In [ ]:
boxplot_data = pd.DataFrame({
    "Demographic Group": ['chn mal', 'chn fem', 'mal mal', 'mal fem', 'ind mal', 'ind fem'],
    "Lower Prevalence": ethnicity_ckd_prevalences_low,
    "Upper Prevalence": ethnicity_ckd_prevalences_high
})

df_melted = boxplot_data.melt(id_vars=["Demographic Group"], 
                              value_vars=["Lower Prevalence", "Upper Prevalence"], 
                              var_name="Bound", value_name="Prevalence")

nphs_prevalence  = [11.8, 13.8, 20.7, 17.2, 17.6, 11.0]
categories = ['chn mal', 'chn fem', 'mal mal', 'mal fem', 'ind mal', 'ind fem']
# Plot
plt.figure(figsize=(8, 5))
sns.set_style("whitegrid")

# Boxplot for CKD prevalence range
sns.boxplot(x="Demographic Group", y="Prevalence", data=df_melted, width=0.5, palette="Blues")

# Overlay the target prevalence as points
sns.stripplot(x=categories, y=nphs_prevalence, color="red", 
              marker="o", size=8, label="Target Prevalence", jitter=False)

# Labels and title
plt.ylim(0, 70)
plt.xlabel("Demographic Group")
plt.ylabel("Prevalence (%)")
plt.title("Prevalence of CKD by Demographic Group")
plt.legend()

# Show plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
categories = ['chn mal', 'chn fem', 'mal mal', 'mal fem', 'ind mal', 'ind fem']
nphs_prevalence  = [11.8, 13.8, 20.7, 17.2, 17.6, 11.0]

# Data
simulated_prevalence = ethnicity_ckd_prevalences[:6]  # Simulated prevalence data=
group_labels = ["Chn Male", "Chn Female", "Malay Male", "Malay Female", "Ind Male", "Ind Female"]
# Create DataFrame and print
df = pd.DataFrame({"Group": group_labels, 
                   "Simulated Prevalence (%)": np.round(ethnicity_ckd_prevalences[:6], 2), 
                   "NPHS Reported Prevalence (%)": nphs_prevalence})
print(df)

# Bar plot parameters
width = 0.2  # Bar width
x = np.arange(len(group_labels))

# Plot
plt.figure(figsize=(8, 5))
bars1 = plt.bar(x - width/1.5, simulated_prevalence, width, label="Simulated Prevalence")
bars2 = plt.bar(x + width/1.5, nphs_prevalence, width, label="NPHS Reported Prevalence")

# Add text labels above the bars
for bar in bars1:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.1f}', 
             ha='center', va='bottom', fontsize=10)
for bar in bars2:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.1f}', 
             ha='center', va='bottom', fontsize=10)

# Labels and formatting
plt.ylim(0, 60)
plt.xlabel("Group")
plt.ylabel("Prevalence (%)")
plt.title("Crude Prevalence (%) of CKD among Aged 18 to 74 Years, Simulation vs NPHS Report")
plt.xticks(x, group_labels)
plt.legend()

# Show plot
plt.show()




In [ ]:
import pandas as pd
import numpy as np

# Define age groups
age_ranges = ["18-39", "40-54", "55-69", "70-74"]

# Initialize lists to store prevalence values
low_prevalence = []
upper_prevalence = []

for a_1, a_2 in [(18, 39), (40, 54), (55, 69), (70, 74)]:
    total_population = 0
    total_ckd_cases = {"low": 0, "upper": 0}
    
    for i in range(8):  # Iterate over 8 ethnicity groups
        ages = age_matrix_vec[i][:, -2]
        stages_col = stage_matrix_ls[i][:, -2]
        age_mask = (ages >= a_1) & (ages <= a_2)
        filtered_stages = stages_col[age_mask]
        
        for label, threshold in zip(["low", "upper"], [0, 20]):
            acr_values = albu_1_mat_storage[i][threshold, :, -2]
            filtered_acr = acr_values[age_mask]
            healthy_mask = (filtered_stages <= 2) & (filtered_acr == 0)
            non_healthy_mask = ~healthy_mask
            total_ckd_cases[label] += np.sum(non_healthy_mask)
        
        total_population += np.sum(age_mask)
    
    low_prevalence.append((total_ckd_cases["low"] / total_population) * 100)
    upper_prevalence.append((total_ckd_cases["upper"] / total_population) * 100)

# Create DataFrame
target_values = [5, 10, 21, 36]
ckd_prevalence_df = pd.DataFrame({
    "Age Range": age_ranges,
    "Low Prevalence": upper_prevalence,
    "Upper Prevalence": low_prevalence,
    "Target": target_values
})

# Display DataFrame
print(ckd_prevalence_df)



  Age Range  Low Prevalence  Upper Prevalence  Target
0     18-39        2.263220          5.263416       5
1     40-54      7.034200         14.318697      10
2     55-69      17.346612         26.188953      21
3     70-74       29.856190         36.588162      36

  Age Range  Low Prevalence  Upper Prevalence  Target
0     18-39        2.353095          4.823845       5
1     40-54       14.966016         17.425828      10
2     55-69       27.582347         28.813559      21
3     70-74       35.233431         35.191747      36

In [ ]:
[5,10,21,36]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Data
data = {
    "Age Range": ["18-39", "40-54", "55-69", "70-74"],
    "Low Prevalence": low_prevalence,
    "Upper Prevalence": upper_prevalence,
    "Target": [5.9, 10.8, 21.6, 36]
}
df = pd.DataFrame(data)

# Plot
plt.figure(figsize=(8, 5))
sns.set_style("white")

# Shaded area for prevalence range
plt.fill_between(df["Age Range"], df["Low Prevalence"], df["Upper Prevalence"], color="lightblue", alpha=0.4, label="Estimated Prevalence")

# Line for target prevalence
plt.plot(df["Age Range"], df["Target"], marker="o", linestyle="--", color="red", label="NPHS 2022 Prevalence")
plt.ylim(0,75)
# Labels and title
plt.xlabel("Age Range")
plt.ylabel("Prevalence (%)")
plt.title("Prevalence of CKD by Age Group")
plt.legend()
# Show plot
plt.show()
